In [1]:
# Install the libraries
%pip install pandas openpyxl numpy


[notice] A new release of pip available: 22.2.2 -> 23.3.2
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Import libraries
import pandas as pd
import numpy as np

### Import Data

In [3]:
data = pd.read_excel("../CupidAlgoTestData.xlsx") # Read csv
data.head(3) # Output top 3 rows in dataset

,Timestamp,Name (eg. Gildong Hong),UWaterloo Email,UWaterloo Student Number,Instagram ID,Gender,How old are you turning this year? (만 나이),Year of study,Faculty of study,Preferred Language:,...,"Are you comfortable with being affectionate in public? (Hugging, Hand Holding ETC.)",I usually spend ______ on first dates.,You prefer to study at:,Taking care of your appearance is a priority to you,Your partner having a best friend of the opposite gender isn't a problem to you,You are a ....,Do you enjoy ....,Do you prefer ....,My love language is ....,Do you see yourself getting married in the future?
0,2022-03-06 20:17:31.795,Hyo Shin Park,parkhyoshin@uwaterloo.ca,11111111,instaID1,Male,22,3B,Engineering,Fluent in both,...,Yes,Less than $50,Library,No,Yes,Night Owl 🦉,Trying new things,A hearty steak🥩,Quality time,"Sometime, yes"
1,2022-03-06 20:19:10.066,Kyung Hoon Min,minkyunghoon@uwaterloo.ca,11111111,instaID2,Male,22,3B,Mathematics,Fluent in both,...,No,More than $50,Library,Yes,Yes,Night Owl 🦉,Trying new things,A hearty steak🥩,Acts of service,"Sometime, yes"
2,2022-03-06 20:23:30.609,Soon-kyu Lee,leesoonkyu@uwaterloo.ca,11111111,instaID3,Female,20,1A,Science,Fluent in both,...,Yes,Less than $50,Library,Yes,Yes,Night Owl 🦉,Trying new things,A healthy Bibimbap 🥒,Quality time,"Not this life, no"


### Drop Unwanted Columns and Deal Breaker 

In [4]:
# Drop identity information (e.g. Timestamp, Name, email and etc.)
data = data.iloc[:, 5:]

# Drop looking for other gender
data = data.drop('If you chose "Other", please tell us below', axis=1)

In [5]:
deal_breaker = ['[DEAL BREAKER]I am looking for ....',
                '[DEAL BREAKER] I am looking for someone no more than __ years OLDER than me',
                '[DEAL BREAKER] I am looking for someone no more than __ years YOUNGER than me']

In [6]:
# Drop deal breaker columns
data_compare = data.drop(deal_breaker, axis=1)

# Drop gender and age
deal_breaker_dependency = ['Gender', 'How old are you turning this year? (만 나이)']
data_compare = data.drop(deal_breaker_dependency, axis=1)

In [7]:
data_compare.head(3)

,Year of study,Faculty of study,Preferred Language:,What is your MBTI?,[DEAL BREAKER]I am looking for ....,[DEAL BREAKER] I am looking for someone no more than __ years OLDER than me,[DEAL BREAKER] I am looking for someone no more than __ years YOUNGER than me,[DEAL BREAKER] Where would you like your match located?,I am looking for a ....,You and your partner have come to a disagreement. You're the type to:,...,"Are you comfortable with being affectionate in public? (Hugging, Hand Holding ETC.)",I usually spend ______ on first dates.,You prefer to study at:,Taking care of your appearance is a priority to you,Your partner having a best friend of the opposite gender isn't a problem to you,You are a ....,Do you enjoy ....,Do you prefer ....,My love language is ....,Do you see yourself getting married in the future?
0,3B,Engineering,Fluent in both,INFP,Female,5,2,Canada,Casual relationship [Lets see where this goes],Confront your partner and express your thoughts,...,Yes,Less than $50,Library,No,Yes,Night Owl 🦉,Trying new things,A hearty steak🥩,Quality time,"Sometime, yes"
1,3B,Mathematics,Fluent in both,ENTJ,Female,2,3,Canada,Casual relationship [Lets see where this goes],Let it slide unless they ask,...,No,More than $50,Library,Yes,Yes,Night Owl 🦉,Trying new things,A hearty steak🥩,Acts of service,"Sometime, yes"
2,1A,Science,Fluent in both,ENFP,Male,3,2,Canada,Casual relationship [Lets see where this goes],Confront your partner and express your thoughts,...,Yes,Less than $50,Library,Yes,Yes,Night Owl 🦉,Trying new things,A healthy Bibimbap 🥒,Quality time,"Not this life, no"


### Initialize Match Matrix

In [9]:
participants_count = len(data)

# Initilze match matrix
MM = np.zeros((participants_count, participants_count))
MM

array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])

In [10]:
# Example: Compare a==person1 vs. b==person2
a = np.array(data_compare.iloc[1])
b = np.array(data_compare.iloc[2])
a == b

array([False, False,  True, False, False, False, False,  True,  True,
       False,  True,  True,  True, False, False,  True, False,  True,
        True, False,  True, False, False,  True,  True,  True,  True,
        True, False, False, False])

In [11]:
# Adjust by weight of question (length = 31 in this case)
# Can be negative
# Leave this for now
weights = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10,
                    1, 2, 3, 4, 5, 6, 7, 8, 9, 10,
                    1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 1])
print(f"Matching value for person1 and person2 is: {sum(weights * (a == b))}")

Matching value for person1 and person2 is: 80


### Run Matching Matrix algorithm

In [12]:
for i in range(participants_count):
    curr_person = np.array(data_compare.iloc[i]) # Person to be compared
    for j in range(participants_count):
        if i != j:
            new_person = np.array(data_compare.iloc[j]) # New person to be compared
            MM[i][j] = sum(weights * (curr_person == new_person)) # Update Matching Matrix

In [13]:
MM 

array([[  0.,  74., 106.,  98.,  86.,  89.,  87., 112.,  75.,  88.],
       [ 74.,   0.,  80.,  63., 102.,  70., 104.,  96.,  94., 103.],
       [106.,  80.,   0.,  57.,  99., 140.,  88., 117.,  95., 107.],
       [ 98.,  63.,  57.,   0.,  67.,  62.,  72.,  93.,  77.,  97.],
       [ 86., 102.,  99.,  67.,   0.,  98., 110., 103.,  92.,  89.],
       [ 89.,  70., 140.,  62.,  98.,   0.,  86.,  98., 105., 107.],
       [ 87., 104.,  88.,  72., 110.,  86.,   0., 108.,  89., 113.],
       [112.,  96., 117.,  93., 103.,  98., 108.,   0.,  96., 137.],
       [ 75.,  94.,  95.,  77.,  92., 105.,  89.,  96.,   0., 120.],
       [ 88., 103., 107.,  97.,  89., 107., 113., 137., 120.,   0.]])

### Deal with Deal Breaker for Gender***

In [24]:
data['[DEAL BREAKER]I am looking for ....'] #responses

0    Female
1    Female
2      Male
3    Female
4    Female
5      Male
6    Female
7    Female
8      Male
9      Male
Name: [DEAL BREAKER]I am looking for ...., dtype: object

In [25]:
data['Gender'] # Gender

0      Male
1      Male
2    Female
3      Male
4      Male
5    Female
6      Male
7      Male
8    Female
9    Female
Name: Gender, dtype: object

In [ ]:
### Write code here ###
# Sort out deal breaker
# If answer for '[DEAL BREAKER]I am looking for ....' question is women
# then all men should have matching value of 0 and vice versa.
# Make sure the matching matrix is updated

# Maybe you can loop through like this
for i in range(participants_count):
    for j in range(participants_count):

### Deal with Deal Breaker for Age Range

In [ ]:
deal_breaker = ['[DEAL BREAKER]I am looking for ....',
                '[DEAL BREAKER] I am looking for someone no more than __ years OLDER than me',
                '[DEAL BREAKER] I am looking for someone no more than __ years YOUNGER than me']

# Make sure that if someones ages is not inside the range of question 2 or 3 make the matching value 0
# You can try this if you have time

### Deal with Special Questions/Cases

### Run Optimal Matching Algorithm

https://brilliant.org/wiki/hungarian-matching/ 
or just take the maximum value in the matrix and remove it.

https://github.com/tdedecko/hungarian-algorithm/blob/master/hungarian.py

In [11]:
import sys
# change this file path to the appropriate one
sys.path.insert(0, '/Users/jaehojung/Desktop/side_projects/cupids_algorithm/hungarian-algorithm/')
from hungarian import Hungarian

KeyboardInterrupt: 

In [17]:
results = Hungarian(MM*(-1))
results.calculate()
matches = results.get_results()
matches

[(2, 5),
 (3, 0),
 (5, 2),
 (7, 9),
 (9, 7),
 (0, 3),
 (1, 8),
 (8, 1),
 (6, 4),
 (4, 6)]

### Remove duplicates from match***

In [26]:
matches 

### Write code here ###
# remove the duplicates here 
# Maybe you can sort each pair and make sure the list is unique

[(2, 5),
 (3, 0),
 (5, 2),
 (7, 9),
 (9, 7),
 (0, 3),
 (1, 8),
 (8, 1),
 (6, 4),
 (4, 6)]

### Produce results to text file (include pair names and match value)